# The modelling process and problem framing in Python

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/waelrash1/loopbricks/blob/main/notebooks/12-process.ipynb)

This notebook is the Python companion to [The modelling process and problem framing](https://waelrash1.github.io/loopbricks/#/learn/process). A useful model starts from a problem, not from a list of everything that might matter. Here you practise three framing decisions with small models: describing the problem as a **reference mode** (a pattern over time), choosing a **time horizon** long enough to see that pattern, and drawing a **boundary** that keeps the important feedback inside the model.

In [ ]:
# Run this cell first. In Colab it installs the loopbricks package, which takes about ten seconds.
try:
    import loopbricks as lb
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "loopbricks @ git+https://github.com/waelrash1/loopbricks.git@main#subdirectory=python"])
    import loopbricks as lb
import matplotlib.pyplot as plt
print("loopbricks", lb.__version__, "with molecules:", ", ".join(lb.molecules()))

## 1. Start from a reference mode

A reference mode is a sketch of how the troubling quantities behave over time. Take a software project that keeps reporting "almost done". The pattern people complain about is not that the work is slow, but that reported progress races to about 90% and then the last 10% takes as long again.

A dynamic hypothesis is a structure that could produce that pattern. **Rework Cycle** (`reworkCycle`) is one: some work is done wrong, and the errors hide as `Undiscovered rework` until someone finds them, which takes a `Rework discovery time` of 5 weeks on average. Everything in the hidden stock still counts as done in the progress report.

In [ ]:
rw = lb.molecule("reworkCycle")
d = rw.run()
reported = (d["Work done"] + d["Undiscovered rework"]) / 10   # percent of the 1000 tasks
true = d["Work done"] / 10

fig, ax = plt.subplots(figsize=(8, 4))
reported.plot(ax=ax, label="reported progress (%)")
true.plot(ax=ax, label="work really done (%)")
ax.set_xlabel("week"); ax.legend(); ax.grid(alpha=0.3)
print(f"Reported progress passes 90% in week {reported.index[reported >= 90][0]:.1f} and 99% in week {reported.index[reported >= 99][0]:.1f}")
print(f"Work really done passes 99% in week {true.index[true >= 99][0]:.1f}")

The model reproduces the reference mode: the report says 99% by week 13.4, yet the work is not 99% done until week 26.7, twice as late. The gap is the stock of undiscovered rework. That is what a good dynamic hypothesis does: it explains the pattern by a structure you can point to, instead of by people being lazy or optimistic.

**Try it.** If the hypothesis is right, better quality should shrink the long tail. Compare `quality` of 0.6, 0.8 and 0.95. How long does the project really take to reach 99%?

<details><summary>Show a worked answer</summary>

With quality 0.6 the work is 99% done in week 40.5, with 0.8 in week 26.7, and with 0.95 in week 15.8. The reported 90% mark barely moves (week 13.6, 11.4 and 10.3), so the progress report hides almost all of the difference. A test like this, where the hypothesis predicts something you could check against real projects, is how a modeller builds confidence in it.
</details>

In [ ]:
for q in (0.6, 0.8, 0.95):
    d = rw.run(quality=q)
    reported = (d["Work done"] + d["Undiscovered rework"]) / 10
    true = d["Work done"] / 10
    print(f"quality {q}: report says 90% in week {reported.index[reported >= 90][0]:4.1f}, "
          f"really 99% done in week {true.index[true >= 99][0]:4.1f}")

## 2. Choose a time horizon long enough to see the problem

The time horizon should reach far enough back to see how the problem arose and far enough ahead to see the delayed effects of a policy. A horizon that is too short can make the wrong structure look right.

Suppose you only had the first 15 years of a population that follows **Constrained Growth** (`logistic`). The cell fits a straight line to the logarithm of those 15 years, which is the same as fitting steady exponential growth, and then projects it forward.

In [ ]:
import numpy as np

pop = lb.molecule("logistic").run()["Population"]
early = pop[pop.index <= 15]
slope, intercept = np.polyfit(early.index, np.log(early.values), 1)
projected = np.exp(intercept + slope * pop.index)

fig, ax = plt.subplots(figsize=(8, 4))
pop.plot(ax=ax, label="the model, 60 years")
ax.plot(pop.index, projected, ls="--", label="exponential fitted to years 0 to 15")
ax.axvline(15, c="grey", ls=":"); ax.set_ylim(0, 400); ax.set_xlabel("year"); ax.legend(); ax.grid(alpha=0.3)

print(f"Fitted growth: {np.exp(slope) - 1:.1%} a year")
for t in (15, 30, 45):
    print(f"year {t}: model {pop[t]:6.0f}, exponential projection {np.exp(intercept + slope * t):8.0f}")

In the first 15 years the two curves stay close (87 against 96 in year 15), and an exponential of about 16% a year looks like a fair fit. Projected to year 30, it predicts about 860 when the model gives 184, and the error grows from there. The balancing loop was there all along; it was simply too weak to show in a short window.

**Try it.** How long a window do you need before the slowdown becomes visible? A simple check: in which year does the yearly net growth (`Growth − Crowding deaths`) stop rising?

<details><summary>Show a worked answer</summary>

Net growth peaks in year 16.4, when the population is half the carrying capacity. Any window that ends before then shows growth that is still accelerating, which is what exponential growth looks like. You need data well past year 16 to see the bend, which is why a modeller asks what limits the growth before the data shows it.
</details>

In [ ]:
d = lb.molecule("logistic").run()
net = d["Growth"] - d["Crowding deaths"]
print(f"Net growth peaks in year {net.idxmax():.1f}, at a population of {d['Population'][net.idxmax()]:.0f}")

## 3. Draw the boundary around the feedback that matters

Every model leaves things out. The test of a boundary is whether something you treated as a fixed input really responds to what happens inside the model. If it does, and you leave it out, your policy can look good in the model and fail in the world.

**Productivity, Overtime & Fatigue** (`prodFatigue`) is a team working `Overtime` (1.4 means 40% longer hours). Fatigue builds towards the overtime level over a `Time to fatigue` of 4 weeks and cuts productivity. To draw the boundary narrower, so that fatigue is treated as fixed, set the time to fatigue so large that fatigue never moves.

In [ ]:
pf = lb.molecule("prodFatigue")
print("Inside the model (stocks):", [s["name"] for s in pf.stocks])
print("Taken as given (constants):", pf.params)

fig, ax = plt.subplots(figsize=(8, 4))
for label, ttf in [("fatigue inside the boundary", 4), ("fatigue left out", 1e9)]:
    out = pf.run(overtime=1.4, time_to_fatigue=ttf)["Output"]
    out.plot(ax=ax, label=label)
    print(f"{label:<28}: output in week 1 {out[1]:5.1f}, week 60 {out.iloc[-1]:5.1f}, total {out.sum() * 0.1:6.0f}")
ax.axhline(100, c="grey", ls=":"); ax.set_xlabel("week"); ax.set_ylabel("output"); ax.legend(); ax.grid(alpha=0.3)

With fatigue left out, 40% more hours give 40% more output forever: a total of 8414 over 60 weeks. With fatigue inside the boundary, output starts at 140 and slides to 84, below the 100 the team made on normal hours, and the total is 5272. Same policy, opposite conclusion; the only difference is where the boundary was drawn. (The `Output` column is a rate per week, so the total is the sum times `dt = 0.1`.)

**Try it.** With fatigue inside the boundary, which overtime level gives the highest output in the long run? Try values from 0.8 to 1.6.

<details><summary>Show a worked answer</summary>

Normal hours (overtime 1) give the most in the long run: 100. Both directions cost output: 0.9 and 1.1 give 99, 1.2 gives 96, 1.4 gives 84 and 1.6 only 64. In this model, once fatigue has caught up, fatigue equals the overtime level and relative output is `overtime × MIN(1.1, 2 − fatigue)`, which is largest at 1. Short bursts of overtime still help for a few weeks, which is exactly why they are tempting.
</details>

In [ ]:
for ot in (0.8, 0.9, 1, 1.1, 1.2, 1.4, 1.6):
    print(f"overtime {ot}: output at week 60 {pf.run(overtime=ot)['Output'].iloc[-1]:5.1f}")

## Where next

- Back in the app: [The modelling process and problem framing](https://waelrash1.github.io/loopbricks/#/learn/process) walks through the whole cycle, from problem statement to policy tests, with real projects.
- Reference modes are built from the [Patterns of behaviour](https://waelrash1.github.io/loopbricks/#/learn/patterns) you can recognise; testing a model against them is the subject of [Testing and building confidence in models](https://waelrash1.github.io/loopbricks/#/learn/testing).
- Writing the equations for a hypothesis comes next, in [Formulating decision rules and nonlinear effects](https://waelrash1.github.io/loopbricks/#/learn/formulation). For a fuller treatment of the process, see Sterman (2000, ch. 3).